# singular-matrix-mask-trick — faded example 3: Full singular mask pipeline: detect, patch, solve, return validity

> Practice drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `singular-matrix-mask-trick`. The last cell reports your progress on the `Numpy: Singular matrix mask trick` subtopic back to Delta Drills.

**Most of the code is already written — complete the one blanked step**, run the test to check it, then run the last cell to record your progress.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Numpy: Singular matrix mask trick` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`singular-matrix-mask-trick`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "singular-matrix-mask-trick"
DD_SUBTOPIC = "Numpy: Singular matrix mask trick"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

Putting together the full singular mask trick: compute determinants, build is_singular mask, clone and overwrite, solve, and return both solutions and the validity mask. The caller uses `is_valid` to ignore the meaningless solutions from the patched-identity slices. Note that `is_valid = ~is_singular` — True means the original matrix was non-singular.

## Faded exercise 3

Complete `masked_solve(A, b, eps=1e-8)` that applies the full mask trick.

1. Detect singular slices via det.
2. Clone, patch with identity.
3. Solve.
4. Return `(x, is_valid)` where is_valid = ~is_singular.

The blank step is returning both the solution tensor and the validity mask.

**Your task:** complete the one blanked step in the code cell below. The surrounding code, function signatures, and variable names are given — work out the missing expression yourself, then run the test.

In [ ]:
import torch as t

t.manual_seed(0)

def masked_solve(A, b, eps=1e-8):
    K, n, _ = A.shape
    dets = t.linalg.det(A)
    is_singular = dets.abs() < eps
    A_safe = A.clone()
    A_safe[is_singular] = t.eye(n, dtype=A.dtype)
    x = t.linalg.solve(A_safe, b)
    raise NotImplementedError()  # TODO: fill in this step — read the prompt cell above

A = t.zeros(3, 2, 2)
A[0] = t.tensor([[4.0, 1.0], [2.0, 3.0]])
A[1] = t.tensor([[1.0, 1.0], [1.0, 1.0]])  # singular
A[2] = t.tensor([[2.0, 0.0], [0.0, 2.0]])
b = t.tensor([[5.0, 4.0], [3.0, 3.0], [6.0, 2.0]])

x, is_valid = masked_solve(A, b)
print('is_valid:', is_valid.tolist())  # [T, F, T]
for i in range(3):
    if is_valid[i]:
        res = (A[i] @ x[i] - b[i]).abs().max().item()
        print(f'slice {i}: residual={res:.2e}')


def _test():
    import torch as t

    A = t.zeros(3, 2, 2)
    A[0] = t.tensor([[4.0, 1.0], [2.0, 3.0]])
    A[1] = t.tensor([[1.0, 1.0], [1.0, 1.0]])
    A[2] = t.tensor([[2.0, 0.0], [0.0, 2.0]])
    b = t.tensor([[5.0, 4.0], [3.0, 3.0], [6.0, 2.0]])

    x, is_valid = masked_solve(A, b)
    assert is_valid.tolist() == [True, False, True], f'is_valid={is_valid.tolist()}'
    assert x.shape == (3, 2), f'x.shape={x.shape}'
    # valid slices should have near-zero residuals
    for i in [0, 2]:
        res = (A[i] @ x[i] - b[i]).abs().max().item()
        assert res < 1e-4, f'slice {i} residual too large: {res}'


try:
    _test()
    _dd_passed.add('faded3')
    print('[Delta Drills] faded3 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report your progress

Run the cell below to send your progress to Delta Drills. It only counts if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded3'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
import torch as t

t.manual_seed(0)

def masked_solve(A, b, eps=1e-8):
    K, n, _ = A.shape
    dets = t.linalg.det(A)
    is_singular = dets.abs() < eps
    A_safe = A.clone()
    A_safe[is_singular] = t.eye(n, dtype=A.dtype)
    x = t.linalg.solve(A_safe, b)
    return x, ~is_singular

A = t.zeros(3, 2, 2)
A[0] = t.tensor([[4.0, 1.0], [2.0, 3.0]])
A[1] = t.tensor([[1.0, 1.0], [1.0, 1.0]])
A[2] = t.tensor([[2.0, 0.0], [0.0, 2.0]])
b = t.tensor([[5.0, 4.0], [3.0, 3.0], [6.0, 2.0]])

x, is_valid = masked_solve(A, b)
print('is_valid:', is_valid.tolist())
for i in range(3):
    if is_valid[i]:
        res = (A[i] @ x[i] - b[i]).abs().max().item()
        print(f'slice {i}: residual={res:.2e}')
```
</details>